# Multiclass Classifier Ensemble Workflow

This notebook evaluates an ensemble of multiple multiclass classifier checkpoints and can also produce final unlabeled pseudo-labels from the ensemble.

Recommended order:
- train multiple seeds with the seed-specific training configs
- confirm each run produced a `best_model.pt`
- run this notebook to evaluate the averaged ensemble on the held-out test set
- use the ensemble outputs for final unlabeled labeling if the results are strong enough


In [ ]:
from pathlib import Path
import json
import subprocess
import sys

import pandas as pd
from IPython.display import display

cwd = Path.cwd().resolve()
candidate_roots = [cwd, *cwd.parents]
REPO_ROOT = None
for candidate in candidate_roots:
    if (candidate / "src" / "wafer_defect").exists() and (candidate / "configs").exists():
        REPO_ROOT = candidate
        break

if REPO_ROOT is None:
    raise RuntimeError("Could not locate repo root containing src/wafer_defect and configs/")

SRC_ROOT = REPO_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

NOTEBOOK_DIR = REPO_ROOT / "experiments/classifier/multiclass/x64/ensemble"
TRAINING_DIR = NOTEBOOK_DIR.parent / "training"
TRAINING_ARTIFACT_ROOT = TRAINING_DIR / "artifacts"


In [ ]:
checkpoint_paths = [
    TRAINING_ARTIFACT_ROOT / "multiclass_classifier_50k" / "best_model.pt",
    TRAINING_ARTIFACT_ROOT / "multiclass_classifier_50k_seed07" / "best_model.pt",
    TRAINING_ARTIFACT_ROOT / "multiclass_classifier_50k_seed13" / "best_model.pt",
    TRAINING_ARTIFACT_ROOT / "multiclass_classifier_50k_seed21" / "best_model.pt",
]
output_dir = NOTEBOOK_DIR / "artifacts" / "multiclass_classifier_50k_ensemble"
confidence_threshold = 0.98

for checkpoint_path in checkpoint_paths:
    print(checkpoint_path, checkpoint_path.exists())

print("Output dir:", output_dir)
print("Confidence threshold:", confidence_threshold)


In [ ]:
ensemble_command = [
    sys.executable,
    str(REPO_ROOT / "scripts/classifier/ensemble_multiclass_classifier.py"),
    "--data-config",
    str(TRAINING_DIR / "data_config.toml"),
    "--metadata-csv",
    str(REPO_ROOT / "data/processed/x64/wm811k_multiclass_50k/metadata_labeled_50k.csv"),
    "--output-dir",
    str(output_dir),
    "--min-confidence",
    str(confidence_threshold),
    "--checkpoints",
    *[str(path) for path in checkpoint_paths],
]
print("Running:", " ".join(ensemble_command))
subprocess.run(ensemble_command, cwd=REPO_ROOT, check=True)


In [ ]:
metrics = json.loads((output_dir / "metrics.json").read_text(encoding="utf-8"))
test_predictions = pd.read_csv(output_dir / "test_predictions.csv")
unlabeled_predictions = pd.read_csv(output_dir / "unlabeled_predictions.csv")

print("Ensemble size:", metrics["ensemble_size"])
print("Test accuracy:", metrics["test"]["accuracy"])
print("Test balanced accuracy:", metrics["test"]["balanced_accuracy"])
print("Accepted pseudo-label fraction:", unlabeled_predictions["accepted_for_pseudo_label"].mean())

display(test_predictions.head())
display(unlabeled_predictions.head())
